# CNN+LRao detector: Magnetic signal detection example

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import CNN_LRao_functions as f
%matplotlib widget
device = "cuda" if torch.cuda.is_available() else "cpu"
import warnings
warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning
)

### Load and normalize data

In [ ]:
mat = sp.io.loadmat('FSSP3exer14_1.mat')
data = mat["w"].squeeze()
sigma_mad = 1.483*np.median(np.abs(data-np.median(data)))
data = (data-np.median(data))/sigma_mad
plt.figure()
plt.plot(data)

### Hyperparameters

In [ ]:
config = {
    "optim": ["SGD"],# optimizer pytorch for CNN weights,"Adam","SGD","AdamW"
    "lr": [1e-3], # learning rate optimizer
    "weight_decay": 1e-5, # L2 weight decay optimizer
    "dim_inp": 128, # CNN input dimension (sequence length)
    "psd_method_params": [{"name":"periodogram"}],# PSD estimation method, {"name":"yule","order":5},{"name":"yule","order":7},{"name":"yule","order":10},{"name":"periodogram"}
    "max_iterations": 500, # max number of iterations (batches and epochs, since batchsize = whole data set)
    "da": 1e-2, # step size for computation of Jacobian of mean
    "filt_size":[3], # CNN filter size
    "n_hidden_channels": [20], # CNN number of hidden channels
    "n_conv_layers": [3], # CNN number of layers (including input and output)
    "puffer_sides": 10 # puffer on sides of input layer for calculation of gradient to eliminate edge effects
}

config_list = f.gen_list_models(config) # list of config dicts
K = 4 # number of harmonics
psi0 = 0.1 # normalized fundamental frequency
H = f.H_multi_harmonic(K,psi0,config["dim_inp"]).to(device) # observation matrix train
n_SNR = 20 # number of considered SNR levels
snr = torch.logspace(-3.5,-1.5,n_SNR) # SNR levels
snr = torch.tensor([10**(-2.5)])
# snr = torch.tensor([10**(-31/10)])
# snr = torch.tensor([10**(-21.5/10)])
p_data = np.mean(data**2) # variance of data
amplitudes = torch.sqrt(2*p_data*snr/K) # amplitudes according to SNR levels
n_splits_o = 5 # outer cross validation splits
n_repeats_o = 20 # outer cross validation repeats
n_splits_i = 4 # inner cross validation splits
n_repeats_i = 1 # inner cross validation repeats
n_repeats_phase_net = 10 # evaluation repeats for different phases
data_torch = torch.tensor(data,device=device,dtype=torch.float32)
data_torch_reshaped = f.reshape_data(data_torch,config["dim_inp"])

### Train, validate, and test model

In [ ]:
train_fractions = np.linspace(0.1,1.0,5)

In [ ]:
t0 = []
t1 = []
for train_fraction in train_fractions:
    print(train_fraction)
    t0_, t1_, lambda_nc, lfi, config_best, training_times, training_hyper_times, inference_times = f.nested_cv(data_torch_reshaped,H,amplitudes,config_list,n_splits_o,n_repeats_o,n_splits_i,n_repeats_i,n_repeats_phase_net,train_fraction)
    t0.append(t0_)
    t1.append(t1_)


In [ ]:
t0_RaoCGN,t1_RaoCGN,t0_RaoCGN_clipped,t1_RaoCGN_clipped,t0_RaoWGN,t1_RaoWGN,t0_Rao_adaptive_gn, t1_Rao_adaptive_gn = [],[],[],[],[],[],[],[]
# t0_RaoCGN_clipped,t1_RaoCGN_clipped = [], []
for train_fraction in train_fractions:
    t0_RaoCGN_,t1_RaoCGN_,t0_RaoCGN_clipped_,t1_RaoCGN_clipped_,t0_RaoWGN_,t1_RaoWGN_,t0_Rao_adaptive_gn_, t1_Rao_adaptive_gn_ = f.reference_detection(data_torch_reshaped,H,amplitudes,n_splits_o,n_repeats_o,n_repeats_phase_net,train_fraction)
    t0_RaoCGN.append(t0_RaoCGN_)
    t1_RaoCGN.append(t1_RaoCGN_)
    t0_RaoCGN_clipped.append(t0_RaoCGN_clipped_)
    t1_RaoCGN_clipped.append(t1_RaoCGN_clipped_)
    t0_RaoWGN.append(t0_RaoWGN_)
    t1_RaoWGN.append(t1_RaoWGN_)
    t0_Rao_adaptive_gn.append(t0_Rao_adaptive_gn_)
    t1_Rao_adaptive_gn.append(t1_Rao_adaptive_gn_)

In [ ]:
config_cnn_classifier = {
    "optim": ["AdamW"],# optimizer pytorch for CNN weights,"Adam","SGD","AdamW"
    "lr": [3e-4], # learning rate optimizer
    "weight_decay": 1e-5, # L2 weight decay optimizer
    "dim_inp": 128, # CNN input dimension (sequence length)
    "max_iterations": 500, # max number of iterations (batches and epochs, since batchsize = whole data set)
}
config_cnn_classifier_list = f.gen_list_models(config_cnn_classifier) # list of config dicts
t0_dnn = []
t1_dnn = []
for train_fraction in train_fractions:
    print(train_fraction)
    t0_dnn_, t1_dnn_, _, loss_history, config_best_dnn, training_times, training_hyper_times, inference_times = f.nested_cv_bce(data_torch_reshaped.unsqueeze(1), H, torch.ones(1)*amplitudes[0], config_cnn_classifier_list, n_splits_o, n_repeats_o, n_splits_i, n_repeats_i, n_repeats_phase_net,train_fraction)
    t0_dnn.append(t0_dnn_)
    t1_dnn.append(t1_dnn_)

In [ ]:
mean_aucs = []
mean_aucs_RaoCGN = []
mean_aucs_RaoCGN_clipped = []
mean_aucs_RaoLaplace = []
mean_aucs_Rao_adaptive_t = []
mean_aucs_Rao_adaptive_gn = []
mean_aucs_dnn = []
std_aucs = []
std_aucs_RaoCGN = []
std_aucs_RaoCGN_clipped = []
std_aucs_RaoLaplace = []
std_aucs_Rao_adaptive_t = []
std_aucs_Rao_adaptive_gn = []
std_aucs_dnn = []
for i in range(len(train_fractions)):
    n_independent_test_data = len(data_torch)//config["dim_inp"]
    mean_fpr = np.linspace(0, 1, 1000)
    n_indiv = int(n_independent_test_data*0.7) # has to be smaller than
    n_trials = 1000 #in paper: 1000
    def tprs_aucs(t0,t1):
        tprs = []
        aucs = []
        for i in range(n_trials):
            tpr, fpr = f.calc_roc_statistics(torch.tensor(np.random.choice(t0, n_indiv, replace=False)),torch.tensor(np.random.choice(t1, n_indiv, replace=False)))
            interp_tpr = np.interp(mean_fpr, torch.flip(fpr,dims=[0]), torch.flip(tpr,dims=[0]))
            interp_tpr[0] = 0.0
            tprs.append(interp_tpr)
            aucs.append(f.auroc(fpr,tpr,fpr_max=1))
        mean_tpr = np.mean(tprs, axis=0)
        mean_tpr[-1] = 1.0
        std_tpr = np.std(tprs, axis=0)*(np.sqrt(n_indiv/n_independent_test_data))
        mean_auc = f.auroc(torch.tensor(mean_fpr), torch.tensor(mean_tpr))
        std_auc = np.std(aucs)
        return mean_tpr, std_tpr, mean_auc, std_auc
    mean_tpr, std_tpr, mean_auc, std_auc = tprs_aucs(t0[i].squeeze(),t1[i].squeeze())
    mean_aucs.append(mean_auc)
    std_aucs.append(std_auc)
    mean_tpr, std_tpr, mean_auc, std_auc = tprs_aucs(t0_RaoCGN[i].squeeze(),t1_RaoCGN[i].squeeze())
    mean_aucs_RaoCGN.append(mean_auc)
    std_aucs_RaoCGN.append(std_auc)
    mean_tpr, std_tpr, mean_auc, std_auc = tprs_aucs(t0_RaoCGN_clipped[i].squeeze(),t1_RaoCGN_clipped[i].squeeze())
    mean_aucs_RaoCGN_clipped.append(mean_auc)
    std_aucs_RaoCGN_clipped.append(std_auc)
    mean_tpr, std_tpr, mean_auc, std_auc = tprs_aucs(t0_RaoWGN[i].squeeze(),t1_RaoWGN[i].squeeze())
    mean_aucs_RaoLaplace.append(mean_auc)
    std_aucs_RaoLaplace.append(std_auc)
    mean_tpr, std_tpr, mean_auc, std_auc = tprs_aucs(t0_Rao_adaptive_gn[i].squeeze(),t1_Rao_adaptive_gn[i].squeeze())
    mean_aucs_Rao_adaptive_gn.append(mean_auc)
    std_aucs_Rao_adaptive_gn.append(std_auc)
    mean_tpr, std_tpr, mean_auc, std_auc = tprs_aucs(t0_dnn[i].squeeze(),t1_dnn[i].squeeze())
    mean_aucs_dnn.append(mean_auc)
    std_aucs_dnn.append(std_auc)

In [ ]:
fig = plt.figure(figsize=(6,2.5))
plt.plot(train_fractions*len(data_torch_reshaped)*0.8,mean_aucs,label="CNN+LRao")
plt.fill_between(train_fractions*len(data_torch_reshaped)*0.8,np.array(mean_aucs)-np.array(std_aucs),np.array(mean_aucs)+np.array(std_aucs),alpha=0.215)
plt.plot(train_fractions*len(data_torch_reshaped)*0.8,mean_aucs_Rao_adaptive_gn,label="Rao gen Gauss")
plt.fill_between(train_fractions*len(data_torch_reshaped)*0.8,np.array(mean_aucs_Rao_adaptive_gn)-np.array(std_aucs_Rao_adaptive_gn),np.array(mean_aucs_Rao_adaptive_gn)+np.array(std_aucs_Rao_adaptive_gn),alpha=0.2)
plt.plot(train_fractions*len(data_torch_reshaped)*0.8,mean_aucs_dnn,label="CNN")
plt.fill_between(train_fractions*len(data_torch_reshaped)*0.8,np.array(mean_aucs_dnn)-np.array(std_aucs_dnn),np.array(mean_aucs_dnn)+np.array(std_aucs_dnn),alpha=0.2)
plt.legend()
plt.xlabel("training set size")
plt.ylabel("AUROC")

In [ ]:
# fig.savefig("AUROC_traindatasize.pdf",dpi=600)